# Shoe Sell-Through — Exploratory Data Analysis

**Goal:** understand the target variable distribution, validate features, and establish binary class thresholds before model training.

**Constraints enforced throughout:** only signals available *at time of receipt* are used as features — no sales velocity, markdown rate, or any post-receipt data.

**Binary classes:** `healthy` (hot + moderate, sell_through_rate ≥ 0.45) vs. `at-risk` (slow + dead, sell_through_rate < 0.45).

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

DB = Path("../sellthrough.db")

In [ ]:
conn = sqlite3.connect(DB)
df = pd.read_sql_query("SELECT * FROM sku_sell_through", conn)
conn.close()

df["receive_month"] = pd.to_datetime(df["receive_date"]).dt.month

print(f"SKUs: {len(df)}")
df.head()

---
## 1. Target Variable: Sell-Through Rate Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df["sell_through_rate"], bins=20, edgecolor="k", color="steelblue")
axes[0].axvline(df["sell_through_rate"].median(), color="red", linestyle="--",
                label=f"Median: {df['sell_through_rate'].median():.2f}")
axes[0].set_xlabel("Sell-Through Rate")
axes[0].set_ylabel("# SKUs")
axes[0].set_title("Sell-Through Rate Distribution")
axes[0].xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[0].legend()

sorted_rates = df["sell_through_rate"].sort_values(ascending=False).reset_index(drop=True)
axes[1].bar(range(len(sorted_rates)), sorted_rates, color="steelblue", edgecolor="none")
axes[1].set_xlabel("SKU rank")
axes[1].set_ylabel("Sell-Through Rate")
axes[1].set_title("Sell-Through Rate by SKU (ranked)")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

plt.tight_layout()
plt.show()

df["sell_through_rate"].describe().apply(lambda x: f"{x:.3f}")

---
## 2. Binary Class Labels

EDA showed 0 hot / 2 moderate / 31 slow / 30 dead — 4-class model has no statistical power for the top tiers. Collapsed to binary:

| Class | Original tiers | Threshold |
|---|---|---|
| **healthy** | hot + moderate | sell_through_rate ≥ 0.45 |
| **at-risk** | slow + dead | sell_through_rate < 0.45 |

In [ ]:
# Canonical threshold — must match HEALTHY_THRESHOLD in train_model.py
HEALTHY_THRESHOLD = 0.45

CLASS_ORDER  = ["healthy", "at-risk"]
CLASS_COLORS = {"healthy": "#2ca02c", "at-risk": "#d62728"}

df["class"] = df["sell_through_rate"].apply(
    lambda r: "healthy" if r >= HEALTHY_THRESHOLD else "at-risk"
)

counts = df["class"].value_counts().reindex(CLASS_ORDER)
print(counts.to_string())
print(f"\nImbalance ratio  1 : {counts['at-risk'] // max(counts['healthy'], 1)}")
print(f"Baseline accuracy (all at-risk): {counts['at-risk'] / len(df):.1%}")

fig, ax = plt.subplots(figsize=(5, 4))
counts.plot.bar(ax=ax, color=[CLASS_COLORS[c] for c in CLASS_ORDER],
                edgecolor="k", width=0.5)
ax.set_title(f"Class Distribution (n={len(df)} SKUs)")
ax.set_xlabel("")
ax.set_ylabel("# SKUs")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

---
## 3. Price Tier

Buckets: budget (<$80), mid ($80–$130), premium ($130–$175), luxury ($175+).

In [ ]:
PRICE_ORDER = ["budget", "mid", "premium", "luxury"]
pt_present  = [p for p in PRICE_ORDER if p in df["price_tier"].values]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

df.boxplot(column="sell_through_rate", by="price_tier",
           positions=range(len(pt_present)), ax=axes[0])
axes[0].axhline(HEALTHY_THRESHOLD, color="green", linestyle="--",
                label=f"Healthy threshold ({HEALTHY_THRESHOLD:.0%})")
axes[0].set_xticklabels(pt_present, rotation=0)
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[0].set_title("Sell-Through Rate by Price Tier")
axes[0].set_xlabel("")
plt.sca(axes[0])
plt.title("Sell-Through Rate by Price Tier")
axes[0].legend()

cross = (
    pd.crosstab(df["price_tier"], df["class"])
      .reindex(index=pt_present, columns=CLASS_ORDER, fill_value=0)
)
cross.plot.bar(ax=axes[1], stacked=True,
               color=[CLASS_COLORS[c] for c in CLASS_ORDER],
               edgecolor="k", width=0.6)
axes[1].set_title("Class Composition by Price Tier")
axes[1].set_xlabel("")
axes[1].set_ylabel("# SKUs")
plt.xticks(rotation=0)
axes[1].legend(title="class", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()

print(df.groupby("price_tier")["sell_through_rate"]
        .agg(["count", "mean", "median"])
        .reindex(pt_present).round(3).to_string())

---
## 4. Department

In [ ]:
dept_order = (
    df.groupby("department")["sell_through_rate"]
      .median().sort_values(ascending=False).index.tolist()
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

df.boxplot(column="sell_through_rate", by="department",
           positions=range(len(dept_order)), ax=axes[0])
axes[0].axhline(HEALTHY_THRESHOLD, color="green", linestyle="--", alpha=0.7)
axes[0].set_xticklabels(dept_order, rotation=15, ha="right")
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[0].set_title("Sell-Through Rate by Department")
axes[0].set_xlabel("")
plt.sca(axes[0])
plt.title("Sell-Through Rate by Department")

cross_dept = (
    pd.crosstab(df["department"], df["class"])
      .reindex(index=dept_order, columns=CLASS_ORDER, fill_value=0)
)
cross_dept.plot.bar(ax=axes[1], stacked=True,
                    color=[CLASS_COLORS[c] for c in CLASS_ORDER],
                    edgecolor="k", width=0.6)
axes[1].set_title("Class Composition by Department")
axes[1].set_xlabel("")
axes[1].set_ylabel("# SKUs")
plt.xticks(rotation=15, ha="right")
axes[1].legend(title="class", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()

---
## 5. Vendor

In [ ]:
vendor_order = (
    df.groupby("vendor")["sell_through_rate"]
      .median().sort_values(ascending=False).index.tolist()
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

df.boxplot(column="sell_through_rate", by="vendor",
           positions=range(len(vendor_order)), ax=axes[0])
axes[0].axhline(HEALTHY_THRESHOLD, color="green", linestyle="--", alpha=0.7)
axes[0].set_xticklabels(vendor_order, rotation=0)
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[0].set_title("Sell-Through Rate by Vendor")
axes[0].set_xlabel("")
plt.sca(axes[0])
plt.title("Sell-Through Rate by Vendor")

cross_v = (
    pd.crosstab(df["vendor"], df["class"])
      .reindex(index=vendor_order, columns=CLASS_ORDER, fill_value=0)
)
cross_v.plot.bar(ax=axes[1], stacked=True,
                 color=[CLASS_COLORS[c] for c in CLASS_ORDER],
                 edgecolor="k", width=0.6)
axes[1].set_title("Class Composition by Vendor")
axes[1].set_xlabel("")
axes[1].set_ylabel("# SKUs")
plt.xticks(rotation=0)
axes[1].legend(title="class", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()

---
## 6. Seasonality: Receive Month

In [ ]:
month_labels = {1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun"}
df["receive_month_label"] = df["receive_month"].map(month_labels)
month_order = [month_labels[m] for m in sorted(df["receive_month"].unique())]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

df.groupby("receive_month_label")["sell_through_rate"].mean() \
  .reindex(month_order).plot.bar(ax=axes[0], color="steelblue", edgecolor="k", width=0.6)
axes[0].axhline(HEALTHY_THRESHOLD, color="green", linestyle="--", alpha=0.7)
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[0].set_title("Mean Sell-Through Rate by Receive Month")
axes[0].set_xlabel("")
plt.sca(axes[0])
plt.xticks(rotation=0)

cross_m = (
    pd.crosstab(df["receive_month_label"], df["class"])
      .reindex(index=month_order, columns=CLASS_ORDER, fill_value=0)
)
cross_m.plot.bar(ax=axes[1], stacked=True,
                 color=[CLASS_COLORS[c] for c in CLASS_ORDER],
                 edgecolor="k", width=0.6)
axes[1].set_title("Class Composition by Receive Month")
axes[1].set_xlabel("")
axes[1].set_ylabel("# SKUs")
plt.xticks(rotation=0)
axes[1].legend(title="class", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()

---
## 7. Buy Quantity (Units Received)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

for cls in CLASS_ORDER:
    sub = df[df["class"] == cls]
    ax.scatter(sub["units_received"], sub["sell_through_rate"],
               label=cls, color=CLASS_COLORS[cls],
               alpha=0.75, edgecolors="k", linewidths=0.4)

ax.axhline(HEALTHY_THRESHOLD, color="green", linestyle="--", alpha=0.7,
           label=f"Threshold ({HEALTHY_THRESHOLD:.0%})")
ax.set_xlabel("Units Received (style total)")
ax.set_ylabel("Sell-Through Rate")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_title("Units Received vs. Sell-Through Rate")
ax.legend()
plt.tight_layout()
plt.show()

corr_cols = ["units_received", "avg_retail_price", "receive_month", "sell_through_rate"]
print("Pearson correlations with sell_through_rate:")
print(df[corr_cols].corr()["sell_through_rate"].drop("sell_through_rate").round(3).to_string())

---
## 8. Feature Correlation Matrix

In [ ]:
encode_cols = ["vendor", "department", "price_tier"]
df_enc = df[["sell_through_rate", "avg_retail_price", "units_received",
             "receive_month"] + encode_cols].copy()

for col in encode_cols:
    df_enc[col] = df_enc[col].astype("category").cat.codes

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df_enc.corr(), annot=True, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.5, ax=ax)
ax.set_title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

---
## 9. Class Balance & Modeling Notes

In [ ]:
counts = df["class"].value_counts().reindex(CLASS_ORDER)
n = len(df)

print("=== Class distribution ===")
print(counts.to_string())
print()
print(f"Total SKUs             : {n}")
print(f"Healthy (minority)     : {counts['healthy']} ({counts['healthy']/n:.1%})")
print(f"Imbalance ratio        : 1 : {counts['at-risk'] // max(counts['healthy'], 1)}")
print(f"Baseline accuracy      : {counts['at-risk']/n:.1%}  (predict all at-risk)")
print()
print("Modeling notes:")
print("  - Use Leave-One-Out CV — stratified k-fold can't split 2 healthy samples reliably.")
print("  - Use class_weight='balanced' on all classifiers to counter 1:30 imbalance.")
print("  - Primary metric: recall on healthy class (missing a healthy SKU = lost reorder opportunity).")
print("  - Accuracy is misleading here — a dummy classifier already scores 96.8%.")
print("  - Features: vendor, department, price_tier, avg_retail_price, receive_month, units_received.")